# 132588 — sparse-grid point generation to GENE parameters files

**What this does.** Seed 132588 profiles + EQDSK → sparse-grid sampler proposes
points in three scaling axes → each point is reconstructed through CHEASE-BS into
its *own* EQDSK (filename carries the transform) and its *own* iterdb → each pair
is written into its *own* GENE parameters file under a temp directory.

**What this does not do.** Nothing is submitted. No GENE runs. No growth rates.
`_default_submitter` and `_job_finished` stay unexercised — the only never-run
campaign function this touches is `_write_parameters`.

**Read this before running.** With no GENE runs there is no QoI, and the grid
will not produce a second batch until the first is told. To get more than the
single seed point out of the sampler, this notebook tells it **fabricated**
values (`dummy_qoi`). The `session.json` it leaves behind is meaningless as a
scan and must never be resumed by a real campaign. Set `STEPS = 1` if you want
only the one honest point.

Run on NERSC — CHEASE-BS needs the compiled binary via
`TPED/config/user_config.yaml`.

## 1. Inputs you have to provide

These are NERSC-side files. Fill in every one that is `None`, then run the check
cell below and do not continue until it prints `all inputs present`.

In [ ]:
# ---------------------------------------------------------------- REQUIRED

# Baseline GENE parameters file this scan perturbs. Neither this repo nor TPED
# has one for 132588 — high_triangularity/132588 holds only analysis notebooks.
# The 129015 one is the shape assumed here (magn_geometry='tracer_efit', EQDSK
# geomfile, profiles via iterdb_file, x0 set to the analysis radius):
#   TPED/data/discharges/NSTX129015/r_0.85_NE/scanfiles0000/parameters
BASE_PARAMETERS = None

# The seed discharge. Two ways in — pick ONE and leave the other None.
#
#   SEED_DIRPATH  : directory holding 132588's EQDSK + pfile (or GENE profiles).
#                   Auto-discovered, same as the CHEASE-BS canary. PREFERRED:
#                   a pfile carries both rho_tor and rho_pol.
#   SEED_ITERDB   : a 132588 iterdb, converted to GENE profiles files on the way
#                   in. Needs SEED_GFILE alongside it, since CHEASE-BS
#                   reconstructs *from* a source equilibrium and an iterdb has
#                   no geometry. Caveat: an iterdb has rho_tor only, so the
#                   rho_pol column of the converted profiles is filled with
#                   rho_tor.
SEED_DIRPATH = "/global/homes/j/joeschm/data/ST_research/NSTXU_discharges/132588"
SEED_ITERDB  = None
SEED_GFILE   = None            # required only when using SEED_ITERDB

# ---------------------------------------------------------------- OPTIONAL

# Where the generated equilibria and parameters files land. Gitignored.
OUTROOT = "tmp_runs"

# Single k_y for the inner scanlist. The QoI reduction names the same value,
# though nothing is harvested here.
KY = 0.05

# Refinement steps to walk. Batch 0 is ONE point, so 3 steps is roughly 5-10
# equilibria, not 3.
STEPS = 3

# Hard stop on reconstructions. Every one of these is a CHEASE-BS run.
MAX_POINTS = 12

# Prefix for the retagged per-point EQDSKs: g132588_Te1.100-ne0.900-wTe1.200
EQDSK_PREFIX = "g132588"

In [ ]:
import os, sys, json
from datetime import datetime

sys.path.insert(0, os.path.abspath("."))
import pilot_helpers as ph

required = {"BASE_PARAMETERS": BASE_PARAMETERS}
if SEED_ITERDB:
    required.update({"SEED_ITERDB": SEED_ITERDB, "SEED_GFILE": SEED_GFILE})
else:
    required.update({"SEED_DIRPATH": SEED_DIRPATH})

print("inputs:")
missing = ph.check_inputs(required)

if BASE_PARAMETERS and os.path.exists(BASE_PARAMETERS):
    notes = ph.check_template(BASE_PARAMETERS)
    print("\ntemplate check:")
    for n in notes:
        print(f"  ! {n}")
    if not notes:
        print("  nothing to flag")

## 2. Workdir and the dummy-session warning

Everything this notebook writes goes under one timestamped directory, including
the poisoned `session.json`.

In [ ]:
WORKDIR = os.path.abspath(os.path.join(OUTROOT, datetime.now().strftime("%Y%m%d_%H-%M-%S")))
os.makedirs(WORKDIR, exist_ok=True)

with open(os.path.join(WORKDIR, "WARNING-DUMMY-SESSION.txt"), "w") as f:
    f.write("session.json here was advanced with fabricated QoI values by the\n"
            "132588 parameter-generation notebook. The sampler state is\n"
            "meaningless. Do not resume a real campaign from this directory.\n")

print(WORKDIR)

## 3. Load the seed discharge

In [ ]:
if SEED_ITERDB:
    print("seed: iterdb + gfile")
    discharge = ph.seed_from_iterdb(SEED_ITERDB, SEED_GFILE, WORKDIR)
else:
    print("seed: directory auto-discovery")
    discharge = ph.seed_from_dirpath(SEED_DIRPATH)

print("  gfile   ", discharge.gfile_filepath)
print("  pfile   ", discharge.pfile_filepath)
print("  profiles", discharge.profiles_filepaths)

## 4. Sanity-check the nominal mtanh fit

The scan axes are **scale factors on this fit**, so a bad fit does not produce a
bad point — it silently redefines what every axis value means. Check
`rms_relative` and look at the pedestal parameters before spending CHEASE-BS
runs.

In [ ]:
from TPED.projects.discharge_tools.src.discharge_physics import DischargePhysics
from TPED.projects.discharge_tools.src.transforms.mtanh_transforms import fit_mtanh
from TPED.projects.GENE_pipelines.src.point_reconstruction import (
    PILOT_BOUNDS_132588, MTANH_AXES, MTANH_FIT_KWARGS, nominal_fits,
    is_equilibrium_axis)

phys0 = DischargePhysics(discharge)

for var in sorted({v for v, _ in MTANH_AXES.values()}):
    profile, record = fit_mtanh(phys0.ds, var, **MTANH_FIT_KWARGS)
    print(f"{var}:  rms_relative = {record['rms_relative']:.4f}")
    # fit_params is MtanhProfile.as_dict() -- a dict, not a sequence.
    for k, v in record["fit_params"].items():
        print(f"    {k:<10} {v: .6g}")

print("\naxes and bounds (scale factors on the fit above):")
for name, (lo, hi) in PILOT_BOUNDS_132588.items():
    var, kwarg = MTANH_AXES[name]
    print(f"  {name:<16} {var:<3} {kwarg:<13} [{lo}, {hi}]")

Plot the seed pedestal against the fit, and against the corners of the box —
this is the cheapest way to see whether ±30% produces profiles you would
actually want CHEASE-BS to try.

In [ ]:
import matplotlib.pyplot as plt

fits = nominal_fits(phys0, set(PILOT_BOUNDS_132588))
corners = [
    ("nominal",  {}),
    ("Te low",   {"Te_ped_scale": PILOT_BOUNDS_132588["Te_ped_scale"][0]}),
    ("Te high",  {"Te_ped_scale": PILOT_BOUNDS_132588["Te_ped_scale"][1]}),
    ("wTe low",  {"Te_width_scale": PILOT_BOUNDS_132588["Te_width_scale"][0]}),
    ("wTe high", {"Te_width_scale": PILOT_BOUNDS_132588["Te_width_scale"][1]}),
]

fig, ax = plt.subplots(figsize=(7, 4))
for label, point in corners:
    from TPED.projects.GENE_pipelines.src.point_reconstruction import _apply_axes
    p = _apply_axes(phys0, point, fits=fits) if point else phys0
    ax.plot(p.rhot, p.Te, label=label, lw=1.5)
ax.set_xlim(0.8, 1.0); ax.set_xlabel(r"$\rho_{tor}$"); ax.set_ylabel("Te (eV)")
ax.legend(fontsize=8); ax.set_title("Te pedestal at the box corners")
plt.tight_layout(); plt.show()

## 5. Build the sampler and the campaign

The submitter is replaced with one that writes the parameters file and stops.
Everything before that — the sampler, `reconstruct_point`, the acceptance gate,
the ledger, `_write_parameters` — is the production path.

In [ ]:
from TPED.projects.discharge_tools.src.cheasebs_runner import CheasebsAcceptance
from TPED.projects.GENE_pipelines.src.sparse_scan_driver import SparseScanSession
from TPED.projects.GENE_pipelines.src.scan_campaign import (
    GeneScanCampaign, QoISpec, HARVESTED, PENDING, REJECTED)

# The radii the existing 132588 linear scans sit at: r_0.736 (q=4), r_0.825
# (q=5). The gate checks q here, so a point whose q has moved where the physics
# is read gets rejected rather than silently scanned.
GENE_RADII = (0.736, 0.825)

AXIS_SHORT = {"Te_ped_scale": "Te", "ne_ped_scale": "ne", "Te_width_scale": "wTe"}

sampler = SparseScanSession(PILOT_BOUNDS_132588,
                            state_path=os.path.join(WORKDIR, "session.json"))

campaign = GeneScanCampaign(
    sampler=sampler,
    base_discharge=discharge,
    qoi=QoISpec(quantity="gamma", reduction="at_ky", ky=KY),
    workdir=WORKDIR,
    base_parameters=os.path.abspath(BASE_PARAMETERS),
    acceptance=CheasebsAcceptance.production(analysis_radii=GENE_RADII),
    ky_scanlist=[KY],
)
campaign.submitter = ph.write_parameters_only(campaign)
print("campaign ready")

## 6. Generate

Per step: propose points → reconstruct + gate each through CHEASE-BS → rename
each accepted EQDSK to carry its transform → write its parameters file → tell
the sampler fabricated values so the next step produces new points.

This is the slow cell. Each point is a CHEASE-BS run.

In [ ]:
proposed = 0
for step in range(STEPS):
    accepted = campaign.propose()
    batch = campaign.ledger.batch(step)
    proposed += len(batch)
    print(f"step {step}: {len(batch)} point(s), {len(accepted)} accepted, "
          f"{len(batch) - len(accepted)} rejected")
    for e in batch:
        if e.status == REJECTED:
            print(f"    {e.point_id} REJECTED — {e.note}")

    renamed = ph.retag_eqdsks(campaign, step, EQDSK_PREFIX, AXIS_SHORT)
    for pid, path in renamed.items():
        print(f"    {pid} eqdsk -> {os.path.basename(path)}")

    submitted = campaign.submit_pending()
    stuck = [e for e in campaign.ledger.batch(step) if not e.rundir and e.eqdsk]
    for e in stuck:
        print(f"    {e.point_id} PARAMETERS FAILED — {e.note}")

    # Stop here rather than advancing. submit_pending() records the failure
    # reason in the ledger note, and the DUMMY_QOI update below overwrites that
    # note -- so advancing past a failed write destroys the only diagnostic
    # there is, which is exactly what happened on the 22-27-39 run.
    if stuck:
        print(f"\n{len(stuck)} point(s) reconstructed but produced no parameters "
              f"file. Stopping so the ledger keeps the reason.")
        break

    if proposed >= MAX_POINTS:
        print(f"point cap {MAX_POINTS} reached, stopping"); break
    if not accepted:
        print("nothing accepted this step — stopping rather than spending "
              "CHEASE-BS runs on a box the gate rejects"); break
    if len(accepted) != len(batch):
        print("batch incomplete, so the grid cannot advance; stopping"); break

    for e in campaign.ledger.batch(step):
        campaign.ledger.update(e.point_id, status=HARVESTED,
                               qoi=ph.dummy_qoi(e.point), note="DUMMY_QOI")
    campaign.tell_batch(step)
    print(f"    advanced the grid on {len(submitted)} fabricated value(s)")

## 7. Did it work?

Reads every written parameters file back. The three silent failure modes: a scan
axis written into the namelist as a GENE key, an `iterdb_file` still pointing at
the seed, and a `geomfile` that is not this point's EQDSK.

In [ ]:
problems = ph.verify_parameters(campaign, is_equilibrium_axis)

for e in campaign.ledger.entries.values():
    if not e.rundir:
        continue
    print(f"{e.point_id}  {json.dumps(e.point, sort_keys=True)}")
    print(f"    eqdsk   {e.eqdsk}")
    print(f"    iterdb  {e.iterdb}")
    print(f"    params  {os.path.join(e.rundir, 'parameters')}")

if problems:
    print(f"\n{len(problems)} PROBLEM(S):")
    for pid, msg in problems:
        print(f"  {pid}: {msg}")
else:
    print("\nno problems found in the written parameters files")

Print one parameters file in full — the automated checks only catch what they
were told to look for.

In [ ]:
written = [e for e in campaign.ledger.entries.values() if e.rundir]
if written:
    print(open(os.path.join(written[0].rundir, "parameters")).read())
else:
    print("nothing was written")

## 8. Every generated profile, side by side

Reads each point's `profiles_e` / `profiles_i` back off disk — what CHEASE-BS
was actually handed, not a recomputation of the transform. The check that
matters is the last one: if two points share a profile, the transform did not
take, and every CHEASE-BS run would still have succeeded while the axes scanned
nothing.

In [ ]:
profile_rows = ph.summarize_profiles(campaign, AXIS_SHORT)

In [ ]:
ph.plot_profiles(profile_rows)

## 9. Summary record

In [ ]:
summary = os.path.join(WORKDIR, "pilot_summary.json")
with open(summary, "w") as f:
    json.dump({
        "workdir": WORKDIR,
        "base_parameters": os.path.abspath(BASE_PARAMETERS),
        "seed": {"dirpath": SEED_DIRPATH, "iterdb": SEED_ITERDB,
                 "gfile": SEED_GFILE},
        "axes": PILOT_BOUNDS_132588,
        "ky": KY,
        "analysis_radii": list(GENE_RADII),
        "qoi_values_are_fabricated": True,
        "profiles": [{k: v for k, v in r.items()
                      if k not in ("rhot", "Te", "ne", "Ti")}
                     for r in profile_rows],
        "problems": problems,
        "points": [{"point_id": e.point_id, "batch": e.batch, "point": e.point,
                    "status": e.status, "eqdsk": e.eqdsk, "iterdb": e.iterdb,
                    "rundir": e.rundir, "note": e.note}
                   for e in campaign.ledger.entries.values()],
    }, f, indent=1)
print(summary)